# Rendu du décor « The Chase » sur GPU Kaggle

Activer Internet et GPU dans les paramètres du notebook. Aucune clé ni jeton n'est nécessaire. Les résultats s'affichent une ligne par réglage : réglage ; temps par image ; mémoire GPU.

In [ ]:
import shutil, subprocess
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
    GPU_OK = True
else:
    print("Aucun GPU dans cette session : le rendu se fera sur CPU.")
    GPU_OK = False
print("GPU_OK", GPU_OK)

In [ ]:
import os, subprocess, tarfile, urllib.request
B='/kaggle/working/blender'
if not os.path.exists(B):
    url='https://download.blender.org/release/Blender4.5/blender-4.5.15-linux-x64.tar.xz'
    urllib.request.urlretrieve(url,'/kaggle/working/blender.tar.xz')
    with tarfile.open('/kaggle/working/blender.tar.xz') as t: t.extractall('/kaggle/working/')
    os.rename('/kaggle/working/blender-4.5.15-linux-x64', B)
print('blender ok')

In [ ]:
import subprocess
subprocess.run(['git','clone','--depth','1','https://github.com/Dony1769/moteur-animation-eclipse','/kaggle/working/depot'],check=True)

In [ ]:
import json, os, subprocess, urllib.request
RACINE = "/kaggle/working/depot/assets/ph"
os.makedirs(f"{RACINE}/textures", exist_ok=True)
MODELES = {"namaqualand_cliff_01": "namaqualand_cliff_01", "rock_face_01": "rock_face_01",
           "rock_face_02": "rock_face_02", "sand_rocks_small_01": "sand_rocks_small_01"}
def telecharger(url, dst):
    if not os.path.exists(dst):
        subprocess.run(["curl", "-sS", "-m", "900", "-L", "-o", dst, url], check=True)
for nom in MODELES:
    f = json.load(urllib.request.urlopen(f"https://api.polyhaven.com/files/{nom}"))
    telecharger(f["blend"]["1k"]["blend"]["url"], f"{RACINE}/{nom}.blend") if "1k" in f["blend"] else None
    for court, cle, ext in (("diff", "Diffuse", "jpg"), ("nor_gl", "nor_gl", "exr"), ("rough", "Rough", "exr")):
        telecharger(f[cle]["1k"][ext]["url"], f"{RACINE}/textures/{nom}_{court}_1k.{ext}")
telecharger("https://dl.polyhaven.org/file/ph-assets/HDRIs/exr/1k/belfast_sunset_puresky_1k.exr", f"{RACINE}/belfast_sunset_puresky_1k.exr")
print("assets ok")

In [ ]:
import os, subprocess, time, shutil
os.chdir("/kaggle/working/depot")
GPU_OK = shutil.which("nvidia-smi") is not None
reglages = [(1280, 540, 16), (960, 405, 8), (640, 270, 8), (640, 270, 4)]
for rx, ry, spp in reglages:
    env = dict(os.environ, RX=str(rx), RY=str(ry), SPP=str(spp))
    env["DEVICE"] = "GPU" if GPU_OK else "CPU"
    env["BACKEND"] = "OPTIX"
    t = time.time()
    subprocess.run(["/kaggle/working/blender", "-b", "--factory-startup", "--python", "scenes/chase_a.py", "--", "/kaggle/working/out"], env=env, capture_output=True)
    mem = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() if GPU_OK else "sans GPU"
    print(f"{rx}x{ry} {spp} ech. ; {env['DEVICE']} ; {time.time()-t:.1f} s ; {mem}")